# Avito: гибридные источники кандидатов

Ноутбук оценивает компоненты, добавленные поверх исправленного BM25F:
историю взаимодействий, предсказание микрокатегории, E5 и перенос
локации. Финальный `answer.csv` здесь не создаётся.


In [ ]:
%pip install -q pymorphy3==2.0.6 sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1


In [ ]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import gc
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import pymorphy3
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GroupShuffleSplit

SEED = 42


In [ ]:
from pathlib import Path
import os

# Для локального запуска достаточно изменить PROJECT_DIR или задать
# AVITO_PROJECT_DIR / AVITO_DATA_DIR / AVITO_OUTPUT_DIR.
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/yuriymagus/avito-nlp-interns")
DEFAULT_PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", DEFAULT_PROJECT_DIR)).expanduser().resolve()
DATA_DIR = Path(
    os.environ.get(
        "AVITO_DATA_DIR",
        KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else PROJECT_DIR / "data",
    )
).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("AVITO_OUTPUT_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = (
    "train.parquet",
    "benchmark_queries.parquet",
    "benchmark_items.parquet",
)
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Не найдены файлы {missing_files} в {DATA_DIR}. "
        "Измените DATA_DIR в первой ячейке."
    )

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

print(f"train:   {train.shape}")
print(f"queries: {queries.shape}")
print(f"items:   {items.shape}")


## 1. Валидация на новых формулировках


In [ ]:
def normalize_query_group(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .str.casefold()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


train_query_groups = normalize_query_group(train["search_query"])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)
fit_idx, validation_pool_idx = next(
    splitter.split(train, groups=train_query_groups)
)

fit_events = train.iloc[fit_idx].copy()
validation_pool = train.iloc[validation_pool_idx].copy()
fit_groups = set(train_query_groups.iloc[fit_idx])

validation_unique = (
    validation_pool
    .assign(
        query_group=train_query_groups.iloc[validation_pool_idx].to_numpy()
    )
    .sample(frac=1, random_state=SEED)
    .drop_duplicates("query_group")
    .copy()
)

validation_unique["has_params"] = (
    validation_unique["search_infm_params_text"]
    .astype("string")
    .str.strip()
    .ne("")
)
benchmark_has_params = (
    queries["search_infm_params_text"]
    .astype("string")
    .str.strip()
    .ne("")
)

n_with_params = int(benchmark_has_params.sum())
n_without_params = len(queries) - n_with_params

validation_events = pd.concat(
    [
        validation_unique.loc[validation_unique["has_params"]]
        .sample(n=n_with_params, random_state=SEED),
        validation_unique.loc[~validation_unique["has_params"]]
        .sample(n=n_without_params, random_state=SEED),
    ],
    ignore_index=True,
).sample(frac=1, random_state=SEED).reset_index(drop=True)

assert len(validation_events) == len(queries)
assert validation_events["query_group"].is_unique
assert set(validation_events["query_group"]).isdisjoint(fit_groups)

print(f"fit events:        {len(fit_events):,}")
print(f"validation events: {len(validation_events):,}")


In [ ]:
ITEM_COLUMNS = [column for column in train.columns if column.startswith("item_")]
CORPUS_SIZE = len(items)

train_items = (
    train[ITEM_COLUMNS]
    .drop_duplicates("item_id")
    .copy()
)

validation_target_ids = set(
    validation_events["item_id"].astype("string")
)

required_items = train_items.loc[
    train_items["item_id"].astype("string").isin(validation_target_ids)
]
other_items = train_items.loc[
    ~train_items["item_id"].astype("string").isin(validation_target_ids)
]

n_other = CORPUS_SIZE - len(required_items)
assert n_other >= 0 and len(other_items) >= n_other

# Состав корпуса по-прежнему содержит все таргеты, но его порядок не кодирует таргет.
validation_items = (
    pd.concat(
        [
            required_items,
            other_items.sample(n=n_other, random_state=SEED),
        ],
        ignore_index=True,
    )
    .assign(__stable_item_id=lambda frame: frame["item_id"].astype("string"))
    .sort_values("__stable_item_id", kind="mergesort")
    .drop(columns="__stable_item_id")
    .reset_index(drop=True)
)

validation_corpus_ids = set(validation_items["item_id"].astype("string"))

assert len(validation_items) == CORPUS_SIZE
assert validation_items["item_id"].is_unique
assert validation_target_ids <= validation_corpus_ids
assert all(column.startswith("item_") for column in validation_items.columns)

print(f"валидационный корпус: {len(validation_items):,}")
print(f"обязательных таргетов: {len(required_items):,}")
print("Порядок корпуса: stable item_id; search_* columns отсутствуют")


In [ ]:
TOKEN_PATTERN = re.compile(r"[0-9a-zа-я]+")
CYRILLIC_TOKEN_PATTERN = re.compile(r"^[а-я]+$")
morph = pymorphy3.MorphAnalyzer()


def normalize_lexical_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.casefold().replace("ё", "е")
    return " ".join(TOKEN_PATTERN.findall(text))


@lru_cache(maxsize=500_000)
def lemmatize_token(token: str) -> str:
    if not CYRILLIC_TOKEN_PATTERN.fullmatch(token):
        return token
    return morph.parse(token)[0].normal_form.replace("ё", "е")


def lemmatize_text(value: object) -> str:
    normalized = normalize_lexical_text(value)
    return " ".join(lemmatize_token(token) for token in normalized.split())


def normalize_identifier(values: pd.Series) -> np.ndarray:
    return (
        pd.to_numeric(values, errors="coerce")
        .astype("Int64")
        .astype("string")
        .fillna("<NA>")
        .to_numpy()
    )


def make_item_tie_keys(item_ids: np.ndarray) -> np.ndarray:
    '''Целочисленный ключ сохраняет лексикографический порядок item_id.'''
    item_ids = np.asarray(item_ids, dtype=str)
    order = np.argsort(item_ids, kind="mergesort")
    keys = np.empty(len(item_ids), dtype=np.int64)
    keys[order] = np.arange(len(item_ids), dtype=np.int64)
    return keys


In [ ]:
def idf_from_document_frequency(
    document_frequency: np.ndarray,
    n_documents: int,
) -> np.ndarray:
    return np.log1p(
        (n_documents - document_frequency + 0.5)
        / (document_frequency + 0.5)
    ).astype(np.float32)


def normalize_field_term_frequency(
    count_matrix: csr_matrix,
    b: float = 0.75,
) -> csr_matrix:
    '''Нормализует поле на независимой копии и не меняет count_matrix.'''
    matrix = count_matrix.tocsr(copy=True).astype(np.float32, copy=False)
    field_lengths = np.asarray(matrix.sum(axis=1)).ravel().astype(np.float32)
    average_field_length = float(field_lengths.mean())

    if average_field_length == 0.0:
        return matrix

    row_normalization = 1.0 / (
        1.0 - b + b * field_lengths / average_field_length
    )
    matrix.data *= np.repeat(row_normalization, np.diff(matrix.indptr))
    return matrix


def build_bm25f_matrix(
    title_term_frequency: csr_matrix,
    description_term_frequency: csr_matrix,
    inverse_document_frequency: np.ndarray,
    title_weight: float = 1.0,
    description_weight: float = 1.0 / 3.0,
    k1: float = 1.2,
) -> csr_matrix:
    '''B8: веса title=1, description=1/3.'''
    combined_tf = (
        title_term_frequency.multiply(title_weight)
        + description_term_frequency.multiply(description_weight)
    ).tocsr()
    combined_tf.sum_duplicates()
    combined_tf.eliminate_zeros()

    combined_tf.data = (
        combined_tf.data * (k1 + 1.0)
        / (combined_tf.data + k1)
    )
    combined_tf.data *= inverse_document_frequency[combined_tf.indices]
    return combined_tf


In [ ]:
def select_exact_top_k(
    indices: np.ndarray,
    values: np.ndarray,
    item_tie_keys: np.ndarray,
    k: int,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    '''Детерминированный top-k с учётом равных значений на границе.'''
    if mask is not None:
        indices = indices[mask]
        values = values[mask]

    if len(indices) == 0:
        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.float32),
        )

    order = np.lexsort((item_tie_keys[indices], -values))
    selected = order[: min(k, len(order))]
    return (
        indices[selected].astype(np.int32, copy=False),
        values[selected].astype(np.float32, copy=False),
    )


def retrieve_global_and_location(
    query_vectors: csr_matrix,
    item_vectors: csr_matrix,
    query_locations: np.ndarray,
    item_locations: np.ndarray,
    item_tie_keys: np.ndarray,
    global_k: int = 200,
    location_k: int = 100,
    batch_size: int = 16,
) -> dict[str, list[np.ndarray] | np.ndarray]:
    '''За один расчёт строит глобальную выдачу и выдачу по локации.'''
    result = {
        "global_indices": [],
        "global_scores": [],
        "location_indices": [],
        "location_scores": [],
        "nonzero_counts": np.zeros(query_vectors.shape[0], dtype=np.int32),
    }

    for batch_start in range(0, query_vectors.shape[0], batch_size):
        batch_end = min(batch_start + batch_size, query_vectors.shape[0])
        scores = (query_vectors[batch_start:batch_end] @ item_vectors.T).tocsr()

        for local_row in range(scores.shape[0]):
            query_position = batch_start + local_row
            row_start = scores.indptr[local_row]
            row_end = scores.indptr[local_row + 1]
            indices = scores.indices[row_start:row_end]
            values = scores.data[row_start:row_end]
            result["nonzero_counts"][query_position] = len(indices)

            global_indices, global_scores = select_exact_top_k(
                indices, values, item_tie_keys, global_k
            )
            location_mask = (
                item_locations[indices] == query_locations[query_position]
            )
            location_indices, location_scores = select_exact_top_k(
                indices,
                values,
                item_tie_keys,
                location_k,
                mask=location_mask,
            )

            result["global_indices"].append(global_indices)
            result["global_scores"].append(global_scores)
            result["location_indices"].append(location_indices)
            result["location_scores"].append(location_scores)

    return result


In [ ]:
def build_b8_index(item_frame: pd.DataFrame) -> dict:
    start = time.perf_counter()

    title_texts = [lemmatize_text(value) for value in item_frame["item_title_raw"]]
    description_texts = [
        lemmatize_text(value) for value in item_frame["item_description_raw"]
    ]

    vectorizer = CountVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        dtype=np.float32,
    )

    combined_counts = vectorizer.fit_transform(
        f"{title} {description}"
        for title, description in zip(title_texts, description_texts)
    ).tocsr()

    n_documents, n_terms = combined_counts.shape
    document_frequency = np.bincount(
        combined_counts.indices,
        minlength=n_terms,
    ).astype(np.float32)
    inverse_document_frequency = idf_from_document_frequency(
        document_frequency,
        n_documents,
    )

    title_counts = vectorizer.transform(title_texts).tocsr()
    description_counts = vectorizer.transform(description_texts).tocsr()

    title_before = title_counts.data.copy()
    description_before = description_counts.data.copy()

    title_tf = normalize_field_term_frequency(title_counts, b=0.75)
    description_tf = normalize_field_term_frequency(description_counts, b=0.75)

    assert np.array_equal(title_counts.data, title_before)
    assert np.array_equal(description_counts.data, description_before)

    item_matrix = build_bm25f_matrix(
        title_term_frequency=title_tf,
        description_term_frequency=description_tf,
        inverse_document_frequency=inverse_document_frequency,
        title_weight=1.0,
        description_weight=1.0 / 3.0,
        k1=1.2,
    )

    item_ids = item_frame["item_id"].astype("string").to_numpy()
    item_locations = normalize_identifier(item_frame["item_location_id"])
    item_tie_keys = make_item_tie_keys(item_ids)

    del combined_counts, title_counts, description_counts, title_tf, description_tf
    gc.collect()

    print(
        f"B8 index: {item_matrix.shape}, nnz={item_matrix.nnz:,}, "
        f"время={time.perf_counter() - start:.1f} с"
    )

    return {
        "vectorizer": vectorizer,
        "item_matrix": item_matrix,
        "item_ids": item_ids,
        "item_locations": item_locations,
        "item_tie_keys": item_tie_keys,
    }


def transform_queries(
    query_frame: pd.DataFrame,
    vectorizer: CountVectorizer,
) -> tuple[list[str], csr_matrix]:
    lemma_queries = [lemmatize_text(value) for value in query_frame["search_query"]]
    query_counts = vectorizer.transform(lemma_queries).tocsr()
    query_counts.data.fill(1.0)
    return lemma_queries, query_counts


In [ ]:
validation_b8 = build_b8_index(validation_items)
validation_lemma_queries, validation_query_counts = transform_queries(
    validation_events,
    validation_b8["vectorizer"],
)
validation_query_locations = normalize_identifier(
    validation_events["search_location_id"]
)

start = time.perf_counter()
validation_retrieval = retrieve_global_and_location(
    query_vectors=validation_query_counts,
    item_vectors=validation_b8["item_matrix"],
    query_locations=validation_query_locations,
    item_locations=validation_b8["item_locations"],
    item_tie_keys=validation_b8["item_tie_keys"],
    global_k=200,
    location_k=100,
    batch_size=16,
)
print(f"Исправленный поиск на валидации: {time.perf_counter() - start:.1f} с")


In [ ]:
def calculate_hits(
    rankings: list[np.ndarray],
    item_ids: np.ndarray,
    target_ids: np.ndarray,
    k: int = 50,
) -> np.ndarray:
    return np.fromiter(
        (
            target_id in item_ids[ranking[:k]]
            for target_id, ranking in zip(target_ids, rankings)
        ),
        dtype=bool,
        count=len(target_ids),
    )


def build_fallback_orders(
    item_frame: pd.DataFrame,
    interaction_events: pd.DataFrame,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    '''Резервный порядок: история → рейтинг → отзывы → item_id.'''
    item_ids = item_frame["item_id"].astype("string")
    history = (
        interaction_events["item_id"].astype("string").value_counts()
    )
    frame = pd.DataFrame({
        "item_index": np.arange(len(item_frame), dtype=np.int32),
        "item_id": item_ids.to_numpy(),
        "history": item_ids.map(history).fillna(0).to_numpy(),
        "rating": pd.to_numeric(item_frame["item_rating"], errors="coerce")
        .fillna(-1).to_numpy(),
        "reviews": pd.to_numeric(
            item_frame["item_rating_reviews_count"], errors="coerce"
        ).fillna(-1).to_numpy(),
        "location": normalize_identifier(item_frame["item_location_id"]),
    })
    ordered = frame.sort_values(
        ["history", "rating", "reviews", "item_id"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    global_order = ordered["item_index"].to_numpy(dtype=np.int32)
    assert len(global_order) == len(item_frame)
    assert np.unique(global_order).size == len(item_frame)
    by_location = {
        str(location): group["item_index"].to_numpy(dtype=np.int32)
        for location, group in ordered.groupby("location", sort=False)
    }
    return global_order, by_location


def fuse_local_global_top50(
    global_rankings: list[np.ndarray],
    location_rankings: list[np.ndarray],
    query_locations: np.ndarray,
    global_fallback: np.ndarray,
    location_fallback: dict[str, np.ndarray],
    local_quota: int,
    k: int = 50,
) -> list[np.ndarray]:
    '''Берёт квоту по локации, затем глобальную выдачу и резерв.'''
    final_rankings = []

    global_fallback = np.asarray(global_fallback, dtype=np.int32).ravel()

    for query_position, (global_ranking, local_ranking, query_location) in enumerate(zip(
        global_rankings,
        location_rankings,
        query_locations,
    )):
        selected = []
        seen = set()

        def add_until(indices, max_total):
            for raw_item_index in indices:
                if len(selected) >= max_total:
                    break

                item_index = int(raw_item_index)
                if item_index not in seen:
                    seen.add(item_index)
                    selected.append(item_index)

        add_until(local_ranking, max_total=min(local_quota, k))
        add_until(global_ranking, max_total=k)

        if len(selected) < k:
            add_until(location_fallback.get(str(query_location), ()), max_total=k)
        if len(selected) < k:
            add_until(global_fallback, max_total=k)

        if len(selected) != k:
            raise RuntimeError(
                f"Не удалось заполнить top-{k} для query_position={query_position}: "
                f"selected={len(selected)}, global_nonzero={len(global_ranking)}, "
                f"location_nonzero={len(local_ranking)}, "
                f"global_fallback={len(global_fallback)}, "
                f"location={query_location!r}"
            )

        assert len(set(selected)) == k
        final_rankings.append(np.asarray(selected, dtype=np.int32))

    return final_rankings


In [ ]:
validation_target_ids_array = validation_events["item_id"].astype("string").to_numpy()

global_hits_50 = calculate_hits(
    validation_retrieval["global_indices"],
    validation_b8["item_ids"],
    validation_target_ids_array,
    k=50,
)
location_hits_50 = calculate_hits(
    validation_retrieval["location_indices"],
    validation_b8["item_ids"],
    validation_target_ids_array,
    k=50,
)

validation_global_fallback, validation_location_fallback = build_fallback_orders(
    validation_items,
    fit_events,
)

quota_rows = []
quota_rankings = {}

for local_quota in (20, 30, 40, 50):
    rankings = fuse_local_global_top50(
        global_rankings=validation_retrieval["global_indices"],
        location_rankings=validation_retrieval["location_indices"],
        query_locations=validation_query_locations,
        global_fallback=validation_global_fallback,
        location_fallback=validation_location_fallback,
        local_quota=local_quota,
        k=50,
    )
    hits = calculate_hits(
        rankings,
        validation_b8["item_ids"],
        validation_target_ids_array,
        k=50,
    )
    quota_rankings[local_quota] = rankings
    quota_rows.append({
        "model": f"local {local_quota} + global fill",
        "HitRate@50": hits.mean(),
        "positive_queries": int(hits.sum()),
    })

baseline_results = pd.DataFrame(
    [
        {
            "model": "B8 global",
            "HitRate@50": global_hits_50.mean(),
            "positive_queries": int(global_hits_50.sum()),
        },
        {
            "model": "B8 location-only",
            "HitRate@50": location_hits_50.mean(),
            "positive_queries": int(location_hits_50.sum()),
        },
        *quota_rows,
    ]
).set_index("model")

display(baseline_results.round(4))

best_local_quota = max(
    quota_rows,
    key=lambda row: (row["HitRate@50"], -int(row["model"].split()[1])),
)
best_local_quota = int(best_local_quota["model"].split()[1])
validation_baseline_top50 = quota_rankings[best_local_quota]
validation_baseline_hits = calculate_hits(
    validation_baseline_top50,
    validation_b8["item_ids"],
    validation_target_ids_array,
    k=50,
)

candidate_pool_hits = []
candidate_pool_sizes = []
for target_id, global_ranking, location_ranking in zip(
    validation_target_ids_array,
    validation_retrieval["global_indices"],
    validation_retrieval["location_indices"],
):
    pool = np.union1d(global_ranking[:100], location_ranking[:100])
    candidate_pool_sizes.append(len(pool))
    candidate_pool_hits.append(target_id in validation_b8["item_ids"][pool])

print(f"Выбранная квота local: {best_local_quota}/50")
print(f"Итоговый HitRate@50: {validation_baseline_hits.mean():.4f}")
print(
    "Pool global@100 + location@100: "
    f"recall={np.mean(candidate_pool_hits):.4f}, "
    f"mean candidates={np.mean(candidate_pool_sizes):.1f}"
)
print(
    "Эффект location сохраняется: "
    f"{location_hits_50.mean() - global_hits_50.mean():+.4f} HR@50"
)


CatBoostRanker был проверен отдельно на 8 000 запросах. Он получил
`HitRate@50 = 0.8014` против `0.8026` у простого решения с локальной и глобальной выдачей,
поэтому в дальнейшую цепочку не вошёл.


## 2. История одинаковых запросов


In [ ]:
def normalize_params_text(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .fillna("")
        .str.casefold()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def make_history_key_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame({
        "__query": normalize_query_group(frame["search_query"]).to_numpy(),
        "__location": normalize_identifier(frame["search_location_id"]),
        "__category": normalize_identifier(frame["search_category"]),
        "__delivery": pd.to_numeric(
            frame["search_is_delivery_search"], errors="coerce"
        ).fillna(-1).astype(np.int16).to_numpy(),
        "__params": normalize_params_text(
            frame["search_infm_params_text"]
        ).to_numpy(),
    })

    if "item_id" in frame.columns:
        result["__item_id"] = frame["item_id"].astype("string").to_numpy()

    return result


def build_ranked_history_map(
    history_frame: pd.DataFrame,
    key_columns: list[str],
) -> dict[tuple, list[str]]:
    counts = (
        history_frame
        .groupby(key_columns + ["__item_id"], dropna=False)
        .size()
        .rename("__count")
        .reset_index()
        .sort_values(
            key_columns + ["__count", "__item_id"],
            ascending=[True] * len(key_columns) + [False, True],
            kind="mergesort",
        )
    )

    result = {}
    grouper = key_columns[0] if len(key_columns) == 1 else key_columns

    for key, group in counts.groupby(grouper, sort=False, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        result[key] = group["__item_id"].astype(str).tolist()

    return result


def build_history_maps(
    history_events: pd.DataFrame,
    allowed_item_ids: set[str],
) -> dict[str, dict[tuple, list[str]]]:
    history_frame = make_history_key_frame(history_events)
    history_frame = history_frame.loc[
        history_frame["__item_id"].isin(allowed_item_ids)
    ].copy()

    return {
        "context": build_ranked_history_map(
            history_frame,
            ["__query", "__location", "__category", "__delivery", "__params"],
        ),
        "query_location": build_ranked_history_map(
            history_frame,
            ["__query", "__location"],
        ),
        "query": build_ranked_history_map(
            history_frame,
            ["__query"],
        ),
    }


def get_history_candidates(
    query_frame: pd.DataFrame,
    history_maps: dict[str, dict[tuple, list[str]]],
) -> list[list[str]]:
    keys = make_history_key_frame(query_frame)
    result = []

    for query, location, category, delivery, params in zip(
        keys["__query"],
        keys["__location"],
        keys["__category"],
        keys["__delivery"],
        keys["__params"],
    ):
        context_key = (
            query,
            location,
            category,
            delivery,
            params,
        )
        query_location_key = (query, location)
        query_key = (query,)

        selected = []
        seen = set()

        for source in (
            history_maps["context"].get(context_key, ()),
            history_maps["query_location"].get(query_location_key, ()),
            history_maps["query"].get(query_key, ()),
        ):
            for item_id in source:
                if item_id not in seen:
                    seen.add(item_id)
                    selected.append(item_id)

        result.append(selected)

    return result


In [ ]:
benchmark_b8 = build_b8_index(items)
benchmark_item_id_set = set(items["item_id"].astype(str))
benchmark_history_maps = build_history_maps(train, benchmark_item_id_set)
benchmark_history_candidates = get_history_candidates(queries, benchmark_history_maps)
history_counts = np.asarray([len(row) for row in benchmark_history_candidates])
print(f"Покрытие benchmark историей: {(history_counts > 0).mean():.2%}")


In [ ]:
HISTORY_EVENT_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_category",
    "search_is_delivery_search",
    "search_infm_params_text",
    "item_id",
]

# Не копируем длинные item descriptions: для history experiment они не нужны.
train_for_warm = train[HISTORY_EVENT_COLUMNS].reset_index(drop=True).copy()
train_for_warm["__row_id"] = np.arange(len(train_for_warm), dtype=np.int64)
train_for_warm["__query_group"] = normalize_query_group(
    train_for_warm["search_query"]
).to_numpy()

query_group_sizes = train_for_warm["__query_group"].value_counts()
eligible_warm_events = train_for_warm.loc[
    train_for_warm["__query_group"].map(query_group_sizes).ge(2)
    & train_for_warm["item_id"].astype(str).isin(benchmark_item_id_set)
].copy()

warm_validation_events = (
    eligible_warm_events
    .sample(frac=1, random_state=SEED)
    .drop_duplicates("__query_group")
    .sample(
        n=min(len(queries), eligible_warm_events["__query_group"].nunique()),
        random_state=SEED,
    )
    .reset_index(drop=True)
)

held_row_ids = set(warm_validation_events["__row_id"].astype(int))
warm_history_events = train_for_warm.loc[
    ~train_for_warm["__row_id"].isin(held_row_ids)
].copy()

warm_history_maps = build_history_maps(
    history_events=warm_history_events,
    allowed_item_ids=benchmark_item_id_set,
)
warm_history_candidates = get_history_candidates(
    query_frame=warm_validation_events,
    history_maps=warm_history_maps,
)

print(f"Warm-validation events: {len(warm_validation_events):,}")
print(f"Held rows excluded from history: {len(held_row_ids):,}")
print(
    "Запросов с history candidates: "
    f"{np.mean([len(x) > 0 for x in warm_history_candidates]):.2%}"
)


In [ ]:
warm_lemma_queries, warm_query_counts = transform_queries(
    warm_validation_events,
    benchmark_b8["vectorizer"],
)
warm_query_locations = normalize_identifier(
    warm_validation_events["search_location_id"]
)

start = time.perf_counter()
warm_retrieval = retrieve_global_and_location(
    query_vectors=warm_query_counts,
    item_vectors=benchmark_b8["item_matrix"],
    query_locations=warm_query_locations,
    item_locations=benchmark_b8["item_locations"],
    item_tie_keys=benchmark_b8["item_tie_keys"],
    global_k=200,
    location_k=100,
    batch_size=16,
)

warm_global_fallback, warm_location_fallback = build_fallback_orders(
    items,
    warm_history_events,
)
warm_baseline_top50 = fuse_local_global_top50(
    global_rankings=warm_retrieval["global_indices"],
    location_rankings=warm_retrieval["location_indices"],
    query_locations=warm_query_locations,
    global_fallback=warm_global_fallback,
    location_fallback=warm_location_fallback,
    local_quota=50,
    k=50,
)

print(f"Warm lexical retrieval: {time.perf_counter() - start:.1f} с")


In [ ]:
def inject_history_candidates(
    baseline_rankings: list[np.ndarray],
    history_candidates: list[list[str]],
    item_id_to_index: dict[str, int],
    history_k: int,
    final_k: int = 50,
) -> list[np.ndarray]:
    result = []

    for baseline, historical_item_ids in zip(
        baseline_rankings,
        history_candidates,
    ):
        selected = []
        seen = set()

        for item_id in historical_item_ids[:history_k]:
            item_index = item_id_to_index.get(str(item_id))
            if item_index is not None and item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)

        for raw_item_index in baseline:
            item_index = int(raw_item_index)
            if item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)
            if len(selected) == final_k:
                break

        assert len(selected) == final_k
        result.append(np.asarray(selected, dtype=np.int32))

    return result


benchmark_item_id_to_index = {
    str(item_id): item_index
    for item_index, item_id in enumerate(benchmark_b8["item_ids"])
}
warm_target_ids = warm_validation_events["item_id"].astype(str).to_numpy()
warm_baseline_hits = calculate_hits(
    warm_baseline_top50,
    benchmark_b8["item_ids"],
    warm_target_ids,
    k=50,
)

history_validation_rows = [{
    "history_k": 0,
    "HitRate@50": warm_baseline_hits.mean(),
    "positive_queries": int(warm_baseline_hits.sum()),
    "new_hits": 0,
    "lost_hits": 0,
}]
warm_history_rankings = {}

for history_k in (1, 3, 5, 10, 20):
    rankings = inject_history_candidates(
        baseline_rankings=warm_baseline_top50,
        history_candidates=warm_history_candidates,
        item_id_to_index=benchmark_item_id_to_index,
        history_k=history_k,
        final_k=50,
    )
    hits = calculate_hits(
        rankings,
        benchmark_b8["item_ids"],
        warm_target_ids,
        k=50,
    )
    warm_history_rankings[history_k] = rankings
    history_validation_rows.append({
        "history_k": history_k,
        "HitRate@50": hits.mean(),
        "positive_queries": int(hits.sum()),
        "new_hits": int((hits & ~warm_baseline_hits).sum()),
        "lost_hits": int((~hits & warm_baseline_hits).sum()),
    })

history_validation_results = pd.DataFrame(history_validation_rows).set_index(
    "history_k"
)
display(history_validation_results.round(4))

best_history_k = int(
    history_validation_results["HitRate@50"].idxmax()
)
history_improves = (
    best_history_k > 0
    and history_validation_results.loc[best_history_k, "HitRate@50"]
    > history_validation_results.loc[0, "HitRate@50"]
)

print(f"Лучший history_k: {best_history_k}")
print(f"History улучшает warm-validation: {history_improves}")


## 3. Предсказание микрокатегории


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import FeatureUnion


def build_query_intent_texts(frame: pd.DataFrame) -> list[str]:
    queries_text = frame["search_query"].fillna("").astype(str)
    params_text = frame["search_infm_params_text"].fillna("").astype(str)
    return (queries_text + " params " + params_text).tolist()


def make_microcategory_vectorizer() -> FeatureUnion:
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        preprocessor=lemmatize_text,
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.997,
        max_features=250_000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        preprocessor=normalize_lexical_text,
        lowercase=False,
        ngram_range=(3, 5),
        min_df=3,
        max_features=350_000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    return FeatureUnion(
        [
            ("word", word_vectorizer),
            ("char", char_vectorizer),
        ],
        transformer_weights={
            "word": 1.0,
            "char": 0.8,
        },
        n_jobs=1,
    )


def make_microcategory_classifier() -> SGDClassifier:
    return SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=2e-6,
        max_iter=30,
        tol=1e-4,
        average=True,
        random_state=SEED,
        n_jobs=-1,
    )


def top_class_labels(
    classifier: SGDClassifier,
    feature_matrix,
    max_k: int = 10,
) -> np.ndarray:
    scores = classifier.decision_function(feature_matrix)

    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    max_k = min(max_k, scores.shape[1])
    top_positions = np.argsort(
        -scores,
        axis=1,
        kind="stable",
    )[:, :max_k]
    return classifier.classes_[top_positions]


In [ ]:
microcat_fit_mask = (
    fit_events["item_microcat_id"].notna()
    & fit_events["search_query"].notna()
)
microcat_fit_events = fit_events.loc[microcat_fit_mask]

microcat_fit_texts = build_query_intent_texts(microcat_fit_events)
microcat_validation_texts = build_query_intent_texts(validation_events)

microcat_vectorizer = make_microcategory_vectorizer()

start = time.perf_counter()
microcat_fit_matrix = microcat_vectorizer.fit_transform(microcat_fit_texts)
microcat_validation_matrix = microcat_vectorizer.transform(
    microcat_validation_texts
)
microcat_vectorization_seconds = time.perf_counter() - start

microcat_fit_targets = normalize_identifier(
    microcat_fit_events["item_microcat_id"]
)
microcat_validation_targets = normalize_identifier(
    validation_events["item_microcat_id"]
)

microcat_classifier = make_microcategory_classifier()

start = time.perf_counter()
microcat_classifier.fit(
    microcat_fit_matrix,
    microcat_fit_targets,
)
microcat_training_seconds = time.perf_counter() - start

validation_predicted_microcats = top_class_labels(
    classifier=microcat_classifier,
    feature_matrix=microcat_validation_matrix,
    max_k=10,
)

microcat_accuracy_rows = []
for top_k in (1, 3, 5, 10):
    hits = np.fromiter(
        (
            target in predicted[:top_k]
            for target, predicted in zip(
                microcat_validation_targets,
                validation_predicted_microcats,
            )
        ),
        dtype=bool,
        count=len(validation_events),
    )
    microcat_accuracy_rows.append({
        "top_k": top_k,
        "accuracy": hits.mean(),
        "positive_queries": int(hits.sum()),
    })

microcat_accuracy = pd.DataFrame(microcat_accuracy_rows).set_index("top_k")

print(f"Microcategory classes: {len(microcat_classifier.classes_):,}")
print(f"Feature matrix: {microcat_fit_matrix.shape}, nnz={microcat_fit_matrix.nnz:,}")
print(f"Vectorization: {microcat_vectorization_seconds:.1f} с")
print(f"Classifier training: {microcat_training_seconds:.1f} с")
display(microcat_accuracy.round(4))


In [ ]:
def build_microcategory_expert_candidates(
    retrieval: dict,
    item_microcategories: np.ndarray,
    predicted_microcategories: np.ndarray,
    predicted_top_k: int,
    strategy: str = "sequential",
) -> list[np.ndarray]:
    if strategy not in {"sequential", "round_robin"}:
        raise ValueError(f"Неизвестная strategy={strategy}")

    result = []

    for query_position, predicted in enumerate(predicted_microcategories):
        local_ranking = retrieval["location_indices"][query_position]
        global_ranking = retrieval["global_indices"][query_position]
        per_category = []

        for microcategory in predicted[:predicted_top_k]:
            category_items = []
            category_seen = set()

            for ranking in (local_ranking, global_ranking):
                matching = ranking[
                    item_microcategories[ranking] == microcategory
                ]

                for raw_item_index in matching:
                    item_index = int(raw_item_index)
                    if item_index not in category_seen:
                        category_seen.add(item_index)
                        category_items.append(item_index)

            per_category.append(category_items)

        if strategy == "sequential":
            selected = [
                item_index
                for category_items in per_category
                for item_index in category_items
            ]
        else:
            # По одному кандидату каждого класса за цикл. Благодаря этому
            # predicted_top_k=3/5 действительно использует не только top-1.
            selected = []
            max_category_size = max(map(len, per_category), default=0)
            for depth in range(max_category_size):
                for category_items in per_category:
                    if depth < len(category_items):
                        selected.append(category_items[depth])

        result.append(np.asarray(selected, dtype=np.int32))

    return result


def inject_index_candidates(
    baseline_rankings: list[np.ndarray],
    expert_candidates: list[np.ndarray],
    expert_k: int,
    final_k: int = 50,
) -> list[np.ndarray]:
    result = []

    for baseline, expert in zip(baseline_rankings, expert_candidates):
        selected = []
        seen = set()

        for raw_item_index in expert[:expert_k]:
            item_index = int(raw_item_index)
            if item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)

        for raw_item_index in baseline:
            item_index = int(raw_item_index)
            if item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)
            if len(selected) == final_k:
                break

        assert len(selected) == final_k
        result.append(np.asarray(selected, dtype=np.int32))

    return result


In [ ]:
validation_item_microcategories = normalize_identifier(
    validation_items["item_microcat_id"]
)
validation_local50_top50 = quota_rankings[50]
validation_local50_hits = calculate_hits(
    validation_local50_top50,
    validation_b8["item_ids"],
    validation_target_ids_array,
    k=50,
)

start = time.perf_counter()
validation_microcat_retrieval = retrieve_global_and_location(
    query_vectors=validation_query_counts,
    item_vectors=validation_b8["item_matrix"],
    query_locations=validation_query_locations,
    item_locations=validation_b8["item_locations"],
    item_tie_keys=validation_b8["item_tie_keys"],
    global_k=1000,
    location_k=500,
    batch_size=16,
)
print(
    "Глубокий validation retrieval для microcategory expert: "
    f"{time.perf_counter() - start:.1f} с"
)

microcat_expert_cache = {}
microcat_retrieval_rows = [{
    "strategy": "baseline",
    "predicted_top_k": 0,
    "expert_k": 0,
    "HitRate@50": validation_local50_hits.mean(),
    "positive_queries": int(validation_local50_hits.sum()),
    "new_hits": 0,
    "lost_hits": 0,
}]
microcat_validation_rankings = {}

for strategy in ("sequential", "round_robin"):
    for predicted_top_k in (1, 3, 5, 10):
        expert_candidates = build_microcategory_expert_candidates(
            retrieval=validation_microcat_retrieval,
            item_microcategories=validation_item_microcategories,
            predicted_microcategories=validation_predicted_microcats,
            predicted_top_k=predicted_top_k,
            strategy=strategy,
        )
        cache_key = (strategy, predicted_top_k)
        microcat_expert_cache[cache_key] = expert_candidates

        for expert_k in (10, 20, 30):
            rankings = inject_index_candidates(
                baseline_rankings=validation_local50_top50,
                expert_candidates=expert_candidates,
                expert_k=expert_k,
                final_k=50,
            )
            hits = calculate_hits(
                rankings,
                validation_b8["item_ids"],
                validation_target_ids_array,
                k=50,
            )
            ranking_key = (strategy, predicted_top_k, expert_k)
            microcat_validation_rankings[ranking_key] = rankings
            microcat_retrieval_rows.append({
                "strategy": strategy,
                "predicted_top_k": predicted_top_k,
                "expert_k": expert_k,
                "HitRate@50": hits.mean(),
                "positive_queries": int(hits.sum()),
                "new_hits": int((hits & ~validation_local50_hits).sum()),
                "lost_hits": int((~hits & validation_local50_hits).sum()),
            })

microcat_retrieval_results = pd.DataFrame(microcat_retrieval_rows)
display(
    microcat_retrieval_results
    .set_index(["strategy", "predicted_top_k", "expert_k"])
    .round(4)
)

candidate_rows = microcat_retrieval_results.loc[
    microcat_retrieval_results["expert_k"].gt(0)
].copy()
candidate_rows = candidate_rows.sort_values(
    ["HitRate@50", "lost_hits", "expert_k", "predicted_top_k"],
    ascending=[False, True, True, True],
    kind="mergesort",
)
best_microcat_config = candidate_rows.iloc[0]
best_microcat_strategy = str(best_microcat_config["strategy"])
best_predicted_top_k = int(best_microcat_config["predicted_top_k"])
best_microcat_expert_k = int(best_microcat_config["expert_k"])
microcat_improves = (
    best_microcat_config["HitRate@50"]
    > validation_local50_hits.mean()
)

print(
    f"Лучший config: strategy={best_microcat_strategy}, "
    f"predicted_top_k={best_predicted_top_k}, "
    f"expert_k={best_microcat_expert_k}"
)
print(f"Microcategory expert улучшает cold validation: {microcat_improves}")


## 4. Векторный поиск E5


In [ ]:
# Если imports отсутствуют в Kaggle image, один раз выполните:
# %pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1

import faiss
import torch
from sentence_transformers import SentenceTransformer

DENSE_MODEL_NAME = "intfloat/multilingual-e5-small"
DENSE_MODEL_REVISION = "fd1525a9fd15316a2d503bf26ab031a61d056e98"
DENSE_MAX_LENGTH = 160
DENSE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DENSE_BATCH_SIZE = 256 if DENSE_DEVICE == "cuda" else 64


def clean_dense_field(values: pd.Series) -> pd.Series:
    return (
        values.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def build_dense_query_texts(frame: pd.DataFrame) -> list[str]:
    query = clean_dense_field(frame["search_query"])
    params = clean_dense_field(frame["search_infm_params_text"])
    return ("query: " + query + ". " + params).tolist()


def build_dense_item_texts(frame: pd.DataFrame) -> list[str]:
    title = clean_dense_field(frame["item_title_raw"])
    params = clean_dense_field(frame["item_infm_params_text"])
    description = clean_dense_field(frame["item_description_raw"])
    return (
        "passage: " + title + ". " + params + ". " + description
    ).tolist()


def encode_dense_texts(
    model: SentenceTransformer,
    texts: list[str],
) -> np.ndarray:
    embeddings = model.encode(
        texts,
        batch_size=DENSE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return np.ascontiguousarray(embeddings, dtype=np.float32)


In [ ]:
def exact_dense_global_search(
    item_embeddings: np.ndarray,
    query_embeddings: np.ndarray,
    item_ids: np.ndarray,
    k: int = 100,
) -> tuple[list[np.ndarray], list[np.ndarray]]:
    # Порядок корпуса не зависит от разметки валидации.
    corpus_order = np.argsort(item_ids, kind="stable")
    ordered_embeddings = np.ascontiguousarray(
        item_embeddings[corpus_order],
        dtype=np.float32,
    )

    index = faiss.IndexFlatIP(item_embeddings.shape[1])
    index.add(ordered_embeddings)
    scores, positions = index.search(query_embeddings, k)

    rankings = [
        corpus_order[row].astype(np.int32, copy=False)
        for row in positions
    ]
    return rankings, [row.astype(np.float32, copy=False) for row in scores]


def exact_dense_location_search(
    item_embeddings: np.ndarray,
    query_embeddings: np.ndarray,
    item_ids: np.ndarray,
    item_locations: np.ndarray,
    query_locations: np.ndarray,
    k: int = 100,
) -> tuple[list[np.ndarray], list[np.ndarray]]:
    rankings = [None] * len(query_embeddings)
    score_rows = [None] * len(query_embeddings)

    for location in pd.unique(query_locations):
        query_positions = np.flatnonzero(query_locations == location)
        item_positions = np.flatnonzero(item_locations == location)

        if len(item_positions) == 0:
            for query_position in query_positions:
                rankings[query_position] = np.empty(0, dtype=np.int32)
                score_rows[query_position] = np.empty(0, dtype=np.float32)
            continue

        location_order = np.argsort(
            item_ids[item_positions],
            kind="stable",
        )
        ordered_items = item_positions[location_order]

        index = faiss.IndexFlatIP(item_embeddings.shape[1])
        index.add(
            np.ascontiguousarray(
                item_embeddings[ordered_items],
                dtype=np.float32,
            )
        )

        local_k = min(k, len(ordered_items))
        scores, positions = index.search(
            np.ascontiguousarray(
                query_embeddings[query_positions],
                dtype=np.float32,
            ),
            local_k,
        )

        for offset, query_position in enumerate(query_positions):
            rankings[query_position] = ordered_items[
                positions[offset]
            ].astype(np.int32, copy=False)
            score_rows[query_position] = scores[offset].astype(
                np.float32,
                copy=False,
            )

    return rankings, score_rows


def fuse_dense_local_global(
    local_rankings: list[np.ndarray],
    global_rankings: list[np.ndarray],
    local_quota: int,
    k: int = 50,
) -> list[np.ndarray]:
    result = []

    for local_ranking, global_ranking in zip(
        local_rankings,
        global_rankings,
    ):
        selected = []
        seen = set()

        for source, limit in (
            (local_ranking, local_quota),
            (global_ranking, k),
        ):
            source_added = 0
            for raw_item_index in source:
                item_index = int(raw_item_index)
                if item_index in seen:
                    continue
                seen.add(item_index)
                selected.append(item_index)
                source_added += 1
                if source_added == limit or len(selected) == k:
                    break
            if len(selected) == k:
                break

        assert len(selected) == k
        result.append(np.asarray(selected, dtype=np.int32))

    return result


In [ ]:
dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    revision=DENSE_MODEL_REVISION,
    device=DENSE_DEVICE,
)
dense_model.max_seq_length = DENSE_MAX_LENGTH

print(f"Dense device: {DENSE_DEVICE}")
print(f"Dense max sequence length: {dense_model.max_seq_length}")

start = time.perf_counter()
validation_dense_item_embeddings = encode_dense_texts(
    dense_model,
    build_dense_item_texts(validation_items),
)
validation_dense_query_embeddings = encode_dense_texts(
    dense_model,
    build_dense_query_texts(validation_events),
)
print(f"Dense encoding validation: {time.perf_counter() - start:.1f} с")

start = time.perf_counter()
validation_dense_global, validation_dense_global_scores = (
    exact_dense_global_search(
        item_embeddings=validation_dense_item_embeddings,
        query_embeddings=validation_dense_query_embeddings,
        item_ids=validation_b8["item_ids"],
        k=100,
    )
)
validation_dense_local, validation_dense_local_scores = (
    exact_dense_location_search(
        item_embeddings=validation_dense_item_embeddings,
        query_embeddings=validation_dense_query_embeddings,
        item_ids=validation_b8["item_ids"],
        item_locations=validation_b8["item_locations"],
        query_locations=validation_query_locations,
        k=100,
    )
)
print(f"Dense exact retrieval: {time.perf_counter() - start:.1f} с")


In [ ]:
validation_current_best = microcat_validation_rankings[
    (best_microcat_strategy, best_predicted_top_k, best_microcat_expert_k)
]
validation_current_hits = calculate_hits(
    validation_current_best,
    validation_b8["item_ids"],
    validation_target_ids_array,
    k=50,
)

dense_rows = [{
    "local_quota": 0,
    "dense_expert_k": 0,
    "dense_standalone_HR@50": np.nan,
    "final_HR@50": validation_current_hits.mean(),
    "positive_queries": int(validation_current_hits.sum()),
    "new_hits": 0,
    "lost_hits": 0,
}]
dense_validation_rankings = {}

for local_quota in (20, 30, 40, 50):
    dense_rankings = fuse_dense_local_global(
        local_rankings=validation_dense_local,
        global_rankings=validation_dense_global,
        local_quota=local_quota,
        k=50,
    )
    dense_hits = calculate_hits(
        dense_rankings,
        validation_b8["item_ids"],
        validation_target_ids_array,
        k=50,
    )

    for dense_expert_k in (5, 10, 20, 30, 40):
        final_rankings = inject_index_candidates(
            baseline_rankings=validation_current_best,
            expert_candidates=dense_rankings,
            expert_k=dense_expert_k,
            final_k=50,
        )
        final_hits = calculate_hits(
            final_rankings,
            validation_b8["item_ids"],
            validation_target_ids_array,
            k=50,
        )

        key = (local_quota, dense_expert_k)
        dense_validation_rankings[key] = final_rankings
        dense_rows.append({
            "local_quota": local_quota,
            "dense_expert_k": dense_expert_k,
            "dense_standalone_HR@50": dense_hits.mean(),
            "final_HR@50": final_hits.mean(),
            "positive_queries": int(final_hits.sum()),
            "new_hits": int((final_hits & ~validation_current_hits).sum()),
            "lost_hits": int((~final_hits & validation_current_hits).sum()),
        })

dense_results = pd.DataFrame(dense_rows)
display(
    dense_results
    .set_index(["local_quota", "dense_expert_k"])
    .round(4)
)

dense_candidates = dense_results.loc[
    dense_results["dense_expert_k"].gt(0)
].sort_values(
    ["final_HR@50", "lost_hits", "dense_expert_k"],
    ascending=[False, True, True],
    kind="mergesort",
)
best_dense_config = dense_candidates.iloc[0]
best_dense_local_quota = int(best_dense_config["local_quota"])
best_dense_expert_k = int(best_dense_config["dense_expert_k"])
dense_improves = (
    best_dense_config["final_HR@50"]
    > validation_current_hits.mean()
)

print(
    f"Лучший dense config: local_quota={best_dense_local_quota}, "
    f"expert_k={best_dense_expert_k}"
)
print(
    f"Final HitRate@50: {best_dense_config['final_HR@50']:.4f}; "
    f"positive queries: {int(best_dense_config['positive_queries'])}"
)
print(f"Dense улучшает текущий pipeline: {dense_improves}")

dense_results.to_csv(
    OUTPUT_DIR / "dense_validation_results.csv",
    index=False,
)
print(OUTPUT_DIR / "dense_validation_results.csv")


## 5. Перенос локации по истории


In [ ]:
LEARNED_LOCATION_MIN_SUPPORT = 50
LEARNED_LOCATION_MIN_SHARE = 0.60
NO_LOCATION_EXPERT = "<NO_LOCATION_EXPERT>"


def predict_learned_item_locations(
    history_events: pd.DataFrame,
    query_frame: pd.DataFrame,
    item_frame: pd.DataFrame,
    min_support: int = LEARNED_LOCATION_MIN_SUPPORT,
    min_share: float = LEARNED_LOCATION_MIN_SHARE,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    history = pd.DataFrame({
        "search_location": normalize_identifier(
            history_events["search_location_id"]
        ),
        "item_location": normalize_identifier(
            history_events["item_location_id"]
        ),
    })
    history = history.loc[
        history["search_location"].ne("<NA>")
        & history["item_location"].ne("<NA>")
    ]

    transition_counts = (
        history
        .groupby(["search_location", "item_location"], sort=False)
        .size()
        .rename("count")
        .reset_index()
    )
    transition_counts["support"] = (
        transition_counts
        .groupby("search_location")["count"]
        .transform("sum")
    )
    dominant = (
        transition_counts
        .sort_values(
            ["search_location", "count", "item_location"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .drop_duplicates("search_location")
        .copy()
    )
    dominant["dominant_share"] = dominant["count"] / dominant["support"]
    dominant_by_search = dominant.set_index("search_location").to_dict("index")

    available_item_locations = set(
        normalize_identifier(item_frame["item_location_id"])
    )
    query_locations = normalize_identifier(query_frame["search_location_id"])

    predictions = []
    eligible = []
    report_rows = []

    for query_position, search_location in enumerate(query_locations):
        stats = dominant_by_search.get(search_location)
        exact_location_available = search_location in available_item_locations

        if stats is None:
            predicted_location = NO_LOCATION_EXPERT
            support = 0
            dominant_share = 0.0
        else:
            predicted_location = str(stats["item_location"])
            support = int(stats["support"])
            dominant_share = float(stats["dominant_share"])

        use_expert = (
            not exact_location_available
            and predicted_location in available_item_locations
            and predicted_location != search_location
            and support >= min_support
            and dominant_share >= min_share
        )

        predictions.append(
            predicted_location if use_expert else NO_LOCATION_EXPERT
        )
        eligible.append(use_expert)
        report_rows.append({
            "query_position": query_position,
            "search_location": search_location,
            "predicted_item_location": predicted_location,
            "exact_location_available": exact_location_available,
            "transition_support": support,
            "dominant_share": dominant_share,
            "eligible": use_expert,
        })

    return (
        np.asarray(predictions, dtype=object),
        np.asarray(eligible, dtype=bool),
        pd.DataFrame(report_rows),
    )


In [ ]:
validation_learned_locations, validation_location_eligible, validation_location_report = (
    predict_learned_item_locations(
        history_events=fit_events,
        query_frame=validation_events,
        item_frame=validation_items,
    )
)

validation_target_locations = normalize_identifier(
    validation_events["item_location_id"]
)
validation_location_report["target_item_location"] = validation_target_locations
validation_location_report["predicted_location_is_target"] = (
    validation_learned_locations == validation_target_locations
)

validation_learned_location_rankings, validation_learned_location_scores = (
    exact_dense_location_search(
        item_embeddings=validation_dense_item_embeddings,
        query_embeddings=validation_dense_query_embeddings,
        item_ids=validation_b8["item_ids"],
        item_locations=validation_b8["item_locations"],
        query_locations=validation_learned_locations,
        k=10,
    )
)
validation_learned_location_candidates = [
    ranking if use_expert else np.empty(0, dtype=np.int32)
    for ranking, use_expert in zip(
        validation_learned_location_rankings,
        validation_location_eligible,
    )
]

validation_dense_best = dense_validation_rankings[
    (best_dense_local_quota, best_dense_expert_k)
]
validation_dense_best_hits = calculate_hits(
    validation_dense_best,
    validation_b8["item_ids"],
    validation_target_ids_array,
    k=50,
)

learned_location_rows = [{
    "expert_k": 0,
    "HitRate@50": validation_dense_best_hits.mean(),
    "positive_queries": int(validation_dense_best_hits.sum()),
    "eligible_queries": int(validation_location_eligible.sum()),
    "eligible_HitRate@50": validation_dense_best_hits[
        validation_location_eligible
    ].mean(),
    "new_hits": 0,
    "lost_hits": 0,
}]
learned_location_validation_rankings = {}

for expert_k in (5, 10):
    rankings = inject_index_candidates(
        baseline_rankings=validation_dense_best,
        expert_candidates=validation_learned_location_candidates,
        expert_k=expert_k,
        final_k=50,
    )
    hits = calculate_hits(
        rankings,
        validation_b8["item_ids"],
        validation_target_ids_array,
        k=50,
    )
    learned_location_validation_rankings[expert_k] = rankings
    learned_location_rows.append({
        "expert_k": expert_k,
        "HitRate@50": hits.mean(),
        "positive_queries": int(hits.sum()),
        "eligible_queries": int(validation_location_eligible.sum()),
        "eligible_HitRate@50": hits[validation_location_eligible].mean(),
        "new_hits": int((hits & ~validation_dense_best_hits).sum()),
        "lost_hits": int((~hits & validation_dense_best_hits).sum()),
    })

learned_location_results = pd.DataFrame(
    learned_location_rows
).set_index("expert_k")
display(learned_location_results.round(4))

eligible_location_accuracy = validation_location_report.loc[
    validation_location_report["eligible"],
    "predicted_location_is_target",
].mean()
print(f"Eligible validation queries: {validation_location_eligible.sum():,}")
print(f"Target-location accuracy на eligible: {eligible_location_accuracy:.2%}")

learned_location_candidates = (
    learned_location_results
    .drop(index=0)
    .reset_index()
    .sort_values(
        ["HitRate@50", "lost_hits", "expert_k"],
        ascending=[False, True, True],
        kind="mergesort",
    )
)
best_learned_location_row = learned_location_candidates.iloc[0]
best_learned_location_k = int(best_learned_location_row["expert_k"])
learned_location_improves = (
    best_learned_location_row["HitRate@50"]
    > validation_dense_best_hits.mean()
)

print(f"Лучший learned-location expert_k: {best_learned_location_k}")
print(
    f"Final HitRate@50: {best_learned_location_row['HitRate@50']:.4f}; "
    f"positive queries: {int(best_learned_location_row['positive_queries'])}"
)
print(f"Learned-location улучшает dense pipeline: {learned_location_improves}")

learned_location_results.reset_index().to_csv(
    OUTPUT_DIR / "learned_location_validation_results.csv",
    index=False,
)
validation_location_report.to_csv(
    OUTPUT_DIR / "learned_location_validation_queries.csv",
    index=False,
)


## Результат последнего прогона

| Конфигурация | HitRate@50 |
|---|---:|
| BM25F: квота по локации 50 | 0.7981 |
| + микрокатегория | 0.8144 |
| + E5 | 0.8454 |
| + перенос локации | **0.8560** |

На закрытом benchmark полная цепочка получила `Recall@50 = 0.799730`.
